# 06 — Feature Engineering

## Purpose
This notebook transforms the clean sales data into **ML-ready datasets** — one CSV file
per branch, each containing daily-level rows with all the features the forecasting model needs.

This is the most important step before modeling. Every decision here (which products to keep,
how to compute rolling windows, how to aggregate) directly impacts model quality.

## What we do
1. Load and clean the data (same steps as previous notebooks)
2. Create holiday columns
3. Filter to the top 83 products per branch (90% cumulative volume rule)
4. Remove non-commercial items
5. Keep only completed sales (filter out cancellations and voids)
6. Drop columns not needed for ML
7. Aggregate to daily level: 1 row per (branch, date, product)
8. Compute rolling window features: qty sold in past 1/3/7/15/30/60/90/180/365 days
9. Export one CSV per branch to `data/processed/`

## Input
`data/intermediate/datanomodifier.csv`

## Output
7 files in `data/processed/`: `branch_carreta.csv`, `branch_credi_club.csv`, etc.

## Run order
Run after `01_data_cleaning.ipynb`. This notebook MUST run before `07_top_products.ipynb`.

In [53]:
import os

# ─── CHANGE THIS FLAG ─────────────────────────────────────────────────
USE_GITHUB  = False   # True = read from GitHub (Colab), False = read from local clone
# ───────────────────────────────────────────────────────────────────────

GITHUB_BASE = "https://media.githubusercontent.com/media/DiegoLarrieta/PanemReto/main"

if USE_GITHUB:
    PROCESSED_DIR    = f"{GITHUB_BASE}/data/processed"
    WEATHER_DIR      = f"{GITHUB_BASE}/data/weather"
    RAW_DIR          = f"{GITHUB_BASE}/data/raw/Complete Data"
    INTERMEDIATE_DIR = None  # not in repo — run 00_data_pipeline.ipynb locally first
else:
    PROJECT_ROOT     = os.path.abspath(os.path.join(os.getcwd(), ".."))
    PROCESSED_DIR    = os.path.join(PROJECT_ROOT, "data", "processed")
    WEATHER_DIR      = os.path.join(PROJECT_ROOT, "data", "weather")
    RAW_DIR          = os.path.join(PROJECT_ROOT, "data", "raw", "Complete Data")
    INTERMEDIATE_DIR = os.path.join(PROJECT_ROOT, "data", "intermediate")

print("INTERMEDIATE_DIR:", INTERMEDIATE_DIR)
print("PROCESSED_DIR:   ", PROCESSED_DIR)

INTERMEDIATE_DIR: /Users/diego/Documents/ChallengeAI/data/intermediate
PROCESSED_DIR:    /Users/diego/Documents/ChallengeAI/data/processed


## Step 1 — Imports and paths

## Step 2 — Load and clean data

Same cleaning block as previous notebooks. We re-apply it here so this notebook
can run independently.

In [54]:
import numpy as np
import pandas as pd


df = pd.read_csv(os.path.join(INTERMEDIATE_DIR, "datanomodifier.csv"), low_memory=False)

for col in ["operating_date", "closing_time", "captured_time"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df["is_modifier"] = (
    df["is_modifier"].astype("string").str.strip().str.lower()
    .map({"true": True, "false": False}).fillna(False).astype(bool)
)
df["quantity"]     = pd.to_numeric(df["quantity"],     errors="coerce").fillna(0)
df["total_ticket"] = pd.to_numeric(df["total_ticket"], errors="coerce").fillna(0)
df["tavg"]         = pd.to_numeric(df["tavg"],         errors="coerce")

df = df[~df["group"].isin(["CAFE Y BEBIDAS CALIENTES", "JUGOS Y BEBIDAS FRIAS"])].copy()
df["sucursal"] = df["sucursal"].replace({
    "Panem - Hotel Kavia N"      : "Panem - Hotel Kavia",
    "Panem - Plaza QIN N"        : "Panem - Plaza QIN",
    "Panem - Hospital Zambrano N": "Panem - Hospital Zambrano",
    "Panem - La Carreta N"       : "Panem - Carreta",
})
df["item"] = df["item"].replace({"SANDWITCH ENSALADA POLLO": "SANDWICH ENSALADA POLLO"})
df["cold_or_warm"] = np.where(df["tavg"] >= 25, "warm", "cold")

print(f"Loaded {len(df):,} rows | {df['sucursal'].nunique()} branches")

Loaded 1,439,071 rows | 7 branches


## Step 3 — Add holiday columns

Holiday flags are added here (before aggregation) so they survive the groupby step.
Each date is classified as 'Festivo Oficial', 'Puente', or 'No holiday'.

In [55]:
holiday_map = {
    ## 2022
    "2022-01-01": "Festivo Oficial",
    "2022-02-07": "Puente",
    "2022-03-21": "Puente",
    "2022-05-01": "Festivo Oficial",
    "2022-09-16": "Puente",
    "2022-11-21": "Puente",
    "2022-12-25": "Festivo Oficial",

    ## 2023
    "2023-01-01": "Festivo Oficial",
    "2023-02-06": "Puente",
    "2023-03-20": "Puente",
    "2023-05-01": "Puente", 
    "2023-09-16": "Festivo Oficial", 
    "2023-11-20": "Puente", 
    "2023-12-25": "Puente",

    ## 2024
    "2024-01-01": "Puente", 
    "2024-02-05": "Puente", 
    "2024-03-18": "Puente",
    "2024-05-01": "Festivo Oficial", 
    "2024-06-02": "Festivo Oficial", 
    "2024-09-16": "Puente",
    "2024-10-01": "Festivo Oficial", 
    "2024-11-18": "Puente", 
    "2024-12-25": "Festivo Oficial",

    ## 2025
    "2025-01-01": "Festivo Oficial",
    "2025-02-03": "Puente",
    "2025-03-17": "Puente",
    "2025-05-01": "Festivo Oficial",
    "2025-09-16": "Festivo Oficial",
    "2025-11-17": "Puente",
    "2025-12-25": "Festivo Oficial",

    ## 2026

    "2026-01-01": "Festivo Oficial",
    "2026-02-02": "Puente",
    "2026-03-16": "Puente",
    "2026-05-01": "Puente",
    "2026-09-16": "Festivo Oficial",
    "2026-11-16": "Puente",
    "2026-12-25": "Puente",
}
holiday_dt_map = {pd.to_datetime(k): v for k, v in holiday_map.items()}

df["operating_date"] = df["operating_date"].dt.normalize()
df["holiday_type"]   = df["operating_date"].map(holiday_dt_map).fillna("No holiday")
df["holidays"]       = df["holiday_type"] != "No holiday"

print("Holiday type counts:")
print(df["holiday_type"].value_counts())

Holiday type counts:
holiday_type
No holiday         1414003
Puente               15668
Festivo Oficial       9400
Name: count, dtype: int64


## Step 4 — Filter to top 83 products (90% volume rule)

**Why 90%?**
The sales distribution follows a power law: a small number of products account for the
vast majority of volume, and hundreds of products sell very rarely.

Modeling rare products is difficult (too few data points) and low-value for stock planning.
We keep only the products that together account for 90% of total volume — which turns out
to be the top 83 products.

Everything else is noise for our forecasting purpose.

In [56]:
# Calculate cumulative volume share per product
prod_volume = (
    df.groupby("item")["quantity"].sum()
      .sort_values(ascending=False)
)
cumulative_pct = prod_volume.cumsum() / prod_volume.sum()

# Find the cutoff: first index where cumulative % >= 90%
cutoff_idx = (cumulative_pct >= 0.90).idxmax()
cutoff_pos = prod_volume.index.get_loc(cutoff_idx) + 1
print(f"Products needed to reach 90% volume: {cutoff_pos}")

top_products = prod_volume.head(83).index.tolist()

rows_before = len(df)
df = df[df["item"].isin(top_products)].copy()
print(f"Rows before filter: {rows_before:,}")
print(f"Rows after filter:  {len(df):,}")
print(f"Unique products:    {df['item'].nunique()}")

Products needed to reach 90% volume: 71
Rows before filter: 1,439,071
Rows after filter:  1,321,214
Unique products:    83


## Step 5 — Remove non-commercial items

- **SUBSIDIO TEC**: An employee subsidy item — it represents internal discounts, not actual
  product demand. Including it would corrupt forecasting.
- **PAN DE MUERTO**: A seasonal product sold only around Día de Muertos (late October / early November).
  Its extreme seasonality makes it an outlier that could distort the model for other products.

In [57]:
items_to_remove = ["SUBSIDIO TEC", "PAN DE MUERTO"]

rows_before = len(df)
df = df[~df["item"].isin(items_to_remove)].copy()
print(f"Removed {rows_before - len(df):,} rows for {items_to_remove}")
print(f"Unique products remaining: {df['item'].nunique()}")

Removed 9,703 rows for ['SUBSIDIO TEC', 'PAN DE MUERTO']
Unique products remaining: 81


## Step 6 — Keep only completed sales

The raw POS data contains multiple action types: `Venta` (sale), cancellations, voids,
and courtesy comps. We only want to predict actual demand — completed sales.

Filtering to `action == 'Venta'` removes all reversals.

In [58]:
rows_before = len(df)
df = df[df["action"] == "Venta"].copy()
print(f"Rows after keeping only Venta: {len(df):,}  (removed {rows_before - len(df):,} non-sale rows)")

Rows after keeping only Venta: 1,287,553  (removed 23,958 non-sale rows)


## Step 7 — Drop columns not needed for ML

The raw data has 47 columns but the model only needs a small subset. We drop everything
that is either:
- A transaction identifier (POS IDs, order IDs)
- A financial metric we're not predicting (prices, taxes, discounts)
- An operational detail (server name, terminal ID)
- Redundant with columns we're keeping

The columns we keep: `sucursal`, `operating_date`, `item`, `quantity`, `day_name`,
`week_number`, `tavg`, `cold_or_warm`, `holiday_type`, `holidays`, `month`

In [59]:
cols_to_drop = [
    "closing_time", "captured_time", "pdv_txn_id", "order_id", "order_type",
    "order_subtype", "table_number", "party_size", "server",
    "terminal", "capture_terminal", "action", "modifier",
    "description", "is_modifier", "unit_price_with_mods", "unit_price",
    "cost_actual", "cost_with_mods", "cost_ideal", "discount",
    "subtotal_ticket", "iva_ticket", "ieps_ticket", "total_ticket",
    "subtotal_item", "iva_item", "ieps_item", "total_item",
    "subtotal_cortesia_cancel", "iva_cortesia_cancel", "ieps_cortesia_cancel", "total_cortesia_cancel",
    "subtotal_anulacion", "iva_anulacion", "ieps_anulacion", "total_anulacion",
    "clave_platillo", "group_type", "group", "day_part", "hour",
    "_source_file"
]
actual_drops = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=actual_drops)

print(f"Remaining columns ({len(df.columns)}):")
print(df.columns.tolist())

Remaining columns (10):
['sucursal', 'operating_date', 'day_name', 'week_number', 'item', 'quantity', 'tavg', 'cold_or_warm', 'holiday_type', 'holidays']


## Step 8 — Aggregate to daily level

Right now each row is one order line item. We want **one row per (branch, date, product)**.

We `groupby` on those three keys and:
- **Sum** `quantity` — total units sold that day
- **Take the first** value for all other columns — they're the same for all rows
  with the same (branch, date, product) combination

In [60]:
first_cols = [c for c in df.columns if c not in ["sucursal", "operating_date", "item", "quantity"]]

agg_dict = {"quantity": "sum"}
agg_dict.update({c: "first" for c in first_cols})

df = df.groupby(["sucursal", "operating_date", "item"], as_index=False).agg(agg_dict)

print(f"Rows after daily aggregation: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.sample(5)

Rows after daily aggregation: 260,571
Columns: ['sucursal', 'operating_date', 'item', 'quantity', 'day_name', 'week_number', 'tavg', 'cold_or_warm', 'holiday_type', 'holidays']


,sucursal,operating_date,item,quantity,day_name,week_number,tavg,cold_or_warm,holiday_type,holidays
21950,Panem - Carreta,2025-01-31,MUFFIN PLATANO VEGANO,2.0,viernes,5,18.2,cold,No holiday,False
169627,Panem - Plaza QIN,2024-01-05,TARTALETA DE NUEZ,3.0,viernes,1,14.9,cold,No holiday,False
241810,Panem - Punto Valle,2024-10-15,FRENCH TOAST,1.0,martes,42,22.4,cold,No holiday,False
49836,Panem - Hospital Zambrano,2023-05-30,MUFFIN PLATANO VEGANO,12.0,martes,22,25.6,warm,No holiday,False
229008,Panem - Punto Valle,2023-12-26,CROISSANT DE SALMÓN,1.0,martes,52,11.9,cold,No holiday,False


## Step 9 — Rolling window features

This is the most important feature engineering step.

**What is a rolling window feature?**
For each product at each branch, we calculate: 'how many units were sold in the past N days?'
These become the model's lagged features — the model learns patterns like:
'if this product sold a lot in the last 7 days, it will probably sell well tomorrow too.'

**Why 9 different windows (1, 3, 7, 15, 30, 60, 90, 180, 365 days)?**
Different products have different demand cycles:
- 1-day lag catches very recent momentum
- 7-day captures weekly seasonality
- 30-day captures monthly patterns
- 365-day captures year-over-year seasonality

**Critical detail — handling sparse products:**
Many products don't sell every single day. If we compute rolling windows only on
days when there WAS a sale, we would miss zeros.

Solution: we reindex each (branch, product) time series to have a row for EVERY calendar day
(filling missing days with quantity=0), compute the rolling windows, then join the results
back to the original rows only.

**Why `shift(1)`?**
The rolling window is shifted by 1 day so that today's sale is NOT included in today's features.
Without the shift we would be 'leaking' the target variable into the features — the model
would know the answer before making a prediction, which is cheating.

In [61]:
df["operating_date"] = pd.to_datetime(df["operating_date"])
df = df.sort_values(["sucursal", "item", "operating_date"]).reset_index(drop=True)

windows = {
    "qty_roll_1"  : 1,
    "qty_roll_3"  : 3,
    "qty_roll_7"  : 7,
    "qty_roll_15" : 15,
    "qty_roll_30" : 30,
    "qty_roll_60" : 60,
    "qty_roll_90" : 90,
    "qty_roll_180": 180,
    "qty_roll_365": 365,
}
lag_days_list = [7, 14, 28]

_day_name_es = {0:"lunes",1:"martes",2:"miércoles",3:"jueves",4:"viernes",5:"sábado",6:"domingo"}

groups = []
for (branch, item), grp in df.groupby(["sucursal", "item"]):
    # Reindex to full date range — inserts 0-quantity rows for missing days
    grp = grp.set_index("operating_date")
    full_range = pd.date_range(grp.index.min(), grp.index.max(), freq="D")
    grp = grp.reindex(full_range)
    grp["quantity"] = grp["quantity"].fillna(0)

    # Compute each rolling window (shift(1) excludes current day)
    for col_name, w in windows.items():
        grp[col_name] = grp["quantity"].shift(1).rolling(window=w, min_periods=1).sum().fillna(0)

    # Compute exact calendar-aware lags
    for n in lag_days_list:
        grp[f"lag_{n}"] = grp["quantity"].shift(n)

    grp["sucursal"] = branch
    grp["item"]     = item
    groups.append(grp)

df_rolled = pd.concat(groups).reset_index().rename(columns={"index": "operating_date"})

roll_cols = list(windows.keys())
lag_cols  = [f"lag_{n}" for n in lag_days_list]

# Keep ALL calendar days — df_rolled already has the full date range per (branch, product).
# Gap days (no actual sale) have quantity=0 and NaN metadata; fill non-weather columns here.
df = df_rolled.copy()

gap = df["day_name"].isna()
df.loc[gap, "day_name"]     = df.loc[gap, "operating_date"].dt.dayofweek.map(_day_name_es)
df.loc[gap, "week_number"]  = df.loc[gap, "operating_date"].dt.isocalendar().week.astype(int)
df.loc[gap, "holiday_type"] = (
    df.loc[gap, "operating_date"].dt.strftime("%Y-%m-%d").map(holiday_map).fillna("No holiday")
)
df.loc[gap, "holidays"] = df.loc[gap, "holiday_type"] != "No holiday"
# tavg and cold_or_warm for gap rows are filled in Step 9b using the weather file

print(f"Rows (full calendar range, quantity=0 for non-sale days): {len(df):,}")
print(f"  of which gap rows (quantity=0, no sale): {gap.sum():,}")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")
df.head(5)

Rows (full calendar range, quantity=0 for non-sale days): 443,169
  of which gap rows (quantity=0, no sale): 182,598
Columns (22): ['operating_date', 'sucursal', 'item', 'quantity', 'day_name', 'week_number', 'tavg', 'cold_or_warm', 'holiday_type', 'holidays', 'qty_roll_1', 'qty_roll_3', 'qty_roll_7', 'qty_roll_15', 'qty_roll_30', 'qty_roll_60', 'qty_roll_90', 'qty_roll_180', 'qty_roll_365', 'lag_7', 'lag_14', 'lag_28']


,operating_date,sucursal,item,quantity,day_name,week_number,tavg,cold_or_warm,holiday_type,holidays,...,qty_roll_7,qty_roll_15,qty_roll_30,qty_roll_60,qty_roll_90,qty_roll_180,qty_roll_365,lag_7,lag_14,lag_28
0,2022-02-22,Panem - Carreta,ADEREZO,1.0,martes,8.0,25.8,warm,No holiday,False,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1,2022-02-23,Panem - Carreta,ADEREZO,0.0,miércoles,8.0,NaN,NaN,No holiday,False,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN
2,2022-02-24,Panem - Carreta,ADEREZO,0.0,jueves,8.0,NaN,NaN,No holiday,False,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN
3,2022-02-25,Panem - Carreta,ADEREZO,0.0,viernes,8.0,NaN,NaN,No holiday,False,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN
4,2022-02-26,Panem - Carreta,ADEREZO,0.0,sábado,8.0,NaN,NaN,No holiday,False,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN


## Step 9b — Fill weather for zero-sale days

Every (branch, product) pair now has a row for every calendar day in its date range —
including days the branch was open but the product sold nothing, and days the branch
was closed (e.g. Carreta on Sundays).

Gap rows have `quantity = 0` but `tavg` and `cold_or_warm` are still NaN because the
reindex step can't invent temperature data. We fill those from the weather file here.

In [62]:
weather_path = os.path.join(WEATHER_DIR, "Clima_limpio.csv")
weather = (
    pd.read_csv(weather_path, parse_dates=["date"])
      .set_index("date")["tavg"]
)

# Monthly average tavg — fallback when a specific date has no weather reading
monthly_tavg = df.groupby(df["operating_date"].dt.month)["tavg"].mean()

# Fill tavg for all gap rows (any day with no sale: closed days, low-demand days, etc.)
gap_weather = df["tavg"].isna()
df.loc[gap_weather, "tavg"] = df.loc[gap_weather, "operating_date"].map(weather)
df.loc[gap_weather, "tavg"] = df.loc[gap_weather, "tavg"].fillna(
    df.loc[gap_weather, "operating_date"].dt.month.map(monthly_tavg)
)
df["cold_or_warm"] = np.where(df["tavg"] >= 25, "warm", "cold")

# month for all rows
df["month"] = df["operating_date"].dt.month

print(f"Weather gap rows filled: {gap_weather.sum():,}")
print(f"Remaining NaN tavg:      {df['tavg'].isna().sum()}")
print(f"Total rows:              {len(df):,}")

Weather gap rows filled: 182,598
Remaining NaN tavg:      0
Total rows:              443,169


## Step 9c — Closure day audit

Confirm that after adding the closure-day rows all branches now show
a full 7-day week in the data.

In [63]:
all_days = {"lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"}

print("=== Day-of-week coverage per branch ===\n")
for branch in sorted(df["sucursal"].unique()):
    present = set(df[df["sucursal"] == branch]["day_name"].str.strip().str.lower().unique())
    missing = all_days - present
    status  = f"⚠ MISSING: {sorted(missing)}" if missing else "✓ all 7 days present"
    print(f"  {branch:<38} {status}")

=== Day-of-week coverage per branch ===

  Panem - Carreta                        ✓ all 7 days present
  Panem - Credi Club                     ✓ all 7 days present
  Panem - Hospital Zambrano              ✓ all 7 days present
  Panem - Hotel Kavia                    ✓ all 7 days present
  Panem - Plaza Nativa                   ✓ all 7 days present
  Panem - Plaza QIN                      ✓ all 7 days present
  Panem - Punto Valle                    ✓ all 7 days present


## Step 9c — Quincena features

In Mexico, paydays (quincenas) fall on the **15th** and the **last day** of each month.
Demand at mall-adjacent branches (Plaza Nativa, QIN, Punto Valle) tends to spike right
after those dates as customers have more disposable income fresh in hand.

We add two features:
- `is_quincena` — 1 on day 15 or last day of month, 0 otherwise
- `days_since_quincena` — days elapsed since the last payday (0 on payday, grows until the next one)

`days_since_quincena` is the correct signal for purchasing power: it is 0 right after
payday (maximum spending ability) and rises as money gets spent down over the cycle.

In [64]:
import calendar as _cal

def _days_since_last_quincena(date):
    day  = date.day
    last = _cal.monthrange(date.year, date.month)[1]

    candidates = []
    # 15th of this month (if already reached)
    if day >= 15:
        candidates.append(date.replace(day=15))
    # Last day of this month (if today is that day)
    if day >= last:
        candidates.append(date.replace(day=last))
    # Last day of previous month (always a valid prior quincena)
    if date.month == 1:
        prev_last = _cal.monthrange(date.year - 1, 12)[1]
        candidates.append(pd.Timestamp(date.year - 1, 12, prev_last))
    else:
        prev_last = _cal.monthrange(date.year, date.month - 1)[1]
        candidates.append(pd.Timestamp(date.year, date.month - 1, prev_last))

    past = [c for c in candidates if c <= date]
    most_recent = max(past)
    return (date - most_recent).days

df["is_quincena"] = (
    (df["operating_date"].dt.day == 15) |
    (df["operating_date"].apply(lambda d: d.day == _cal.monthrange(d.year, d.month)[1]))
).astype(int)

df["days_since_quincena"] = df["operating_date"].apply(_days_since_last_quincena)

print(f"is_quincena days     : {df['is_quincena'].sum():,} ({df['is_quincena'].mean()*100:.1f}%)")
print(f"days_since_quincena  : {df['days_since_quincena'].min()} – {df['days_since_quincena'].max()}")
df[["operating_date","is_quincena","days_since_quincena"]].drop_duplicates("operating_date").head(20)

is_quincena days     : 28,997 (6.5%)
days_since_quincena  : 0 – 15


,operating_date,is_quincena,days_since_quincena
0,2022-02-22,0,7
1,2022-02-23,0,8
2,2022-02-24,0,9
3,2022-02-25,0,10
4,2022-02-26,0,11
5,2022-02-27,0,12
6,2022-02-28,1,0
7,2022-03-01,0,1
8,2022-03-02,0,2
9,2022-03-03,0,3


## Step 10 — Export one CSV per branch

We split the dataframe by branch and save one file per branch to `data/processed/`.

Each file is self-contained: it has all the rows and features for one branch, ready
for a model to train on without needing any other files.

In [65]:
branch_name_map = {
    "Panem - Carreta"           : "branch_carreta.csv",
    "Panem - Credi Club"        : "branch_credi_club.csv",
    "Panem - Hospital Zambrano" : "branch_hospital_zambrano.csv",
    "Panem - Hotel Kavia"       : "branch_hotel_kavia.csv",
    "Panem - Plaza Nativa"      : "branch_plaza_nativa.csv",
    "Panem - Plaza QIN"         : "branch_plaza_qin.csv",
    "Panem - Punto Valle"       : "branch_punto_valle.csv",
}

for branch, filename in branch_name_map.items():
    branch_df = df[df["sucursal"] == branch].copy()
    path = os.path.join(PROCESSED_DIR, filename)
    branch_df.to_csv(path, index=False)
    print(f"{branch:<35} → {len(branch_df):>7,} rows  saved to {filename}")

print(f"\nDone. {len(branch_name_map)} branch files saved to data/processed/")

Panem - Carreta                     →  78,033 rows  saved to branch_carreta.csv
Panem - Credi Club                  →  21,150 rows  saved to branch_credi_club.csv
Panem - Hospital Zambrano           →  70,663 rows  saved to branch_hospital_zambrano.csv
Panem - Hotel Kavia                 →  69,354 rows  saved to branch_hotel_kavia.csv
Panem - Plaza Nativa                →  35,745 rows  saved to branch_plaza_nativa.csv
Panem - Plaza QIN                   →  80,280 rows  saved to branch_plaza_qin.csv
Panem - Punto Valle                 →  87,944 rows  saved to branch_punto_valle.csv

Done. 7 branch files saved to data/processed/


## Summary

After running this notebook, `data/processed/` contains 7 CSV files, one per branch.
Each file has 20 columns:

| Column | Description |
|--------|-------------|
| `sucursal` | Branch name |
| `operating_date` | Sale date |
| `item` | Product name |
| `quantity` | **Target variable** — units sold that day |
| `day_name` | Day of week (Spanish) |
| `week_number` | ISO week |
| `tavg` | Average temperature °C |
| `cold_or_warm` | 'warm' or 'cold' |
| `holiday_type` | Holiday classification |
| `holidays` | Boolean holiday flag |
| `month` | Month number |
| `qty_roll_1..365` | Rolling window features (9 columns) |

**Next step:** `07_top_products.ipynb` — identify the top 5 products per branch and prepare final model-ready data.